In [1]:
%pip install -q langchain langchain-groq langchain-core python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads GROQ_API_KEY from .env if it exists

# Uncomment and paste your key here only if you are not using .env
# os.environ["GROQ_API_KEY"] = "gsk_YOUR_KEY_HERE"

if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError(
        "GROQ_API_KEY is not set. Set it in a .env file or the cell above."
    )

print("✅ GROQ_API_KEY loaded.")

✅ GROQ_API_KEY loaded.


In [2]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage
from langchain_groq import ChatGroq

d:\Practice_Code\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,          # deterministic tool calls
    max_retries=2,
)

print(f"Model: {llm.model_name}")

Model: llama-3.3-70b-versatile


In [9]:
@tool
def add(a: int, b: int) -> int:
    """Add two integers. Only use integers for 'a' and 'b'."""
    return a + b


@tool
def subtract(a: int, b: int) -> int:
    """
    Subtract b from a.

    Args:
        a (int): the number to subtract from
        b (int): the number to subtract

    Returns:
        int: the difference a - b
    """
    return a - b


@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two integers.

    Args:
        a (int): first integer
        b (int): second integer

    Returns:
        int: the product a * b
    """
    return a * b


print("Tools defined:", [add.name, subtract.name, multiply.name])

Tools defined: ['add', 'subtract', 'multiply']


In [10]:
tool_map: dict = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
}

# Quick sanity-checks
assert tool_map["add"].invoke({"a": 3, "b": 2}) == 5
assert tool_map["subtract"].invoke({"a": 10, "b": 4}) == 6
assert tool_map["multiply"].invoke({"a": 3, "b": 7}) == 21

print("All tool tests passed ✅")

All tool tests passed ✅


In [11]:
tools = list(tool_map.values())
llm_with_tools = llm.bind_tools(tools)

print(f"{len(tools)} tools bound to the LLM.")

3 tools bound to the LLM.


In [12]:
query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)]

print("Chat history initialised:", chat_history)

Chat history initialised: [HumanMessage(content='What is 3 + 2?', additional_kwargs={}, response_metadata={})]


In [13]:
response_1: AIMessage = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

print("Response type:", type(response_1).__name__)
print("Tool calls requested:", response_1.tool_calls)

Response type: AIMessage
Tool calls requested: [{'name': 'add', 'args': {'a': 3, 'b': 2}, 'id': '5vy2q3zzd', 'type': 'tool_call'}]


In [14]:
tool_call = response_1.tool_calls[0]   # take the first call

tool_name    = tool_call["name"]       # e.g. "add"
tool_args    = tool_call["args"]       # e.g. {"a": 3, "b": 2}
tool_call_id = tool_call["id"]         # unique ID linking call <-> result

print(f"Tool to call : {tool_name}")
print(f"Arguments    : {tool_args}")
print(f"Call ID      : {tool_call_id}")

Tool to call : add
Arguments    : {'a': 3, 'b': 2}
Call ID      : 5vy2q3zzd


In [15]:
tool_result  = tool_map[tool_name].invoke(tool_args)
tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)

chat_history.append(tool_message)

print(f"Tool result  : {tool_result}")
print(f"ToolMessage  : {tool_message}")

Tool result  : 5
ToolMessage  : content='5' tool_call_id='5vy2q3zzd'


In [16]:
final_response: AIMessage = llm_with_tools.invoke(chat_history)

print("Final answer:", final_response.content)

Final answer: The answer is 5.


In [17]:
class ToolCallingAgent:
    """A simple LangChain agent that supports single and multi-step tool calling."""

    def __init__(self, llm, tools: list, tool_map: dict) -> None:
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map

    def run(self, query: str, max_steps: int = 5) -> str:
        """
        Run the agent on a natural-language query.

        Args:
            query: the user question
            max_steps: safety limit on the number of tool-call iterations

        Returns:
            str: the model's final natural-language answer
        """
        chat_history = [HumanMessage(content=query)]

        for step in range(max_steps):
            response: AIMessage = self.llm_with_tools.invoke(chat_history)
            chat_history.append(response)

            # No tool calls → model is done
            if not response.tool_calls:
                return response.content

            # Execute every tool call the model requested
            for tc in response.tool_calls:
                name   = tc["name"]
                args   = tc["args"]
                tc_id  = tc["id"]

                if name not in self.tool_map:
                    result = f"Error: unknown tool '{name}'"
                else:
                    result = str(self.tool_map[name].invoke(args))

                chat_history.append(ToolMessage(content=result, tool_call_id=tc_id))

        return "[Agent] Max steps reached without a final answer."

In [18]:
agent = ToolCallingAgent(llm, tools=tools, tool_map=tool_map)

queries = [
    "one plus 2",
    "one minus 2",
    "three times two",
    "what is 7 multiplied by 8?",
]

for q in queries:
    answer = agent.run(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

Q: one plus 2
A: The result of 1 + 2 is 3.

Q: one minus 2
A: The result of 1 minus 2 is -1.

Q: three times two
A: The result of three times two is 6.

Q: what is 7 multiplied by 8?
A: The result of 7 multiplied by 8 is 56.

